In [ ]:

# ── 1. Kaggle API credentials ────────────────────────────────────────────────
# Upload your kaggle.json (from https://www.kaggle.com/settings → API → Create New Token)
from google.colab import files
import os

os.makedirs("/root/.config/kaggle", exist_ok=True)

# Uncomment the next two lines on FIRST run to upload your kaggle.json
# uploaded = files.upload()                                      # select kaggle.json
# !cp kaggle.json /root/.config/kaggle/ && chmod 600 /root/.config/kaggle/kaggle.json

# ── 2. Download dataset ──────────────────────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download("adityajn105/flickr8k")
print("Path to dataset files:", path)

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")

print("Path to dataset files:", path)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pickle
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical,plot_model
from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout,add

In [ ]:
Base_dir="/kaggle/input/flicker8k"
working_dir="/kaggle/working/"

In [ ]:
image_name = "429205889_ff5a006311.jpg"
image_id = image_name.split(".")[0]
img_path = os.path.join(path,"Images", image_name)
image = Image.open(img_path)
plt.imshow(image)

In [ ]:
model=VGG16()
model=Model(inputs=model.inputs,outputs=model.layers[-2].output)
print(model.summary())

In [ ]:
features={}
directory=os.path.join(path,"Images") # Changed Base_dir to path
for img_name in tqdm(os.listdir(directory)):
    img_path=directory+"/"+img_name
    image=load_img(img_path,target_size=(224,224))
    image=img_to_array(image)
    image=image.reshape((1,image.shape[0],image.shape[1],image.shape[2]))
    image=preprocess_input(image)
    feature=model.predict(image,verbose=0)
    image_id=img_name.split(".")[0]
    features[image_id]=feature


In [ ]:
with open(os.path.join(path,"captions.txt"),'r') as File:
  next(File)
  captions_file=File.read()

In [ ]:
print(captions_file[:200])

In [ ]:
mapping={}
for line in tqdm(captions_file.split("\n")):
  tokens=line.split(",")
  if len(line)<2:
    continue
  image_id,caption=tokens[0],tokens[1:]
  image_id=image_id.split(".")[0]
  caption=" ".join(caption)
  if image_id not in mapping:
    mapping[image_id]=[]
  mapping[image_id].append(caption)

In [ ]:
def preprocessing(mapping):
  for key,captions in mapping.items():
    for i in range(len(captions)):
      caption=captions[i].lower()
      caption=caption.replace("[^A-Za-z]","")
      caption=caption.replace("\s+"," ")
      caption="startseq "+" ".join([word for word in caption.split() if len(word)>1])+" endseq"
      captions[i]=caption

preprocessing(mapping)

In [ ]:
# for i, (key, value) in enumerate(mapping.items()):
#     if i >= 70:
#         break
#     print(f"Image ID: {key}\nCaptions: {value}\n")

In [ ]:
all_captions = [caption for key in mapping.keys() for caption in mapping[key]]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(caption.split()) for caption in all_captions)

In [ ]:
image_ids = list(mapping.keys())
# split = int(len(image_ids) * 0.90)
train, test = train_test_split(image_ids, test_size=0.1, random_state=42)
# train = image_ids[:split]
# test = image_ids[split:]

In [ ]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = [] , [] ,[]
    n = 0
    while True:
        for key in data_keys:
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    X1.append(features[key][0])
                    X2.append(in_seq)
                    y.append(out_seq)
            n += 1
            if n == batch_size:
                yield {"image": np.array(X1), "text": np.array(X2)}, np.array(y)
                X1.clear()  # Clears the content of X1
                X2.clear()  # Clears the content of X2
                y.clear()   # Clears the content of y
                n = 0

In [ ]:

inputs1=Input(shape=(4096,),name="image")
a=Dropout(0.3)(inputs1)
X=Dense(256,activation='relu')(a)
inputs2=Input(shape=(max_length,),name="text")
c=Embedding(vocab_size,256)(inputs2)
d=Dropout(0.3)(c)
Y=LSTM(256)(d)
decoder1=add([X , Y])
decoder2=Dense(256,activation='relu')(decoder1)
outputs=Dense(vocab_size,activation='softmax')(decoder2)
model=Model(inputs=[inputs1,inputs2],outputs=outputs)
model.compile(loss='categorical_crossentropy',optimizer='adam')
model.summary()

In [ ]:
tf.keras.utils.plot_model(
    model,
    to_file='model.png',
    show_shapes=True,
    show_dtype=False,
    show_layer_names=True,
    dpi=55,
    show_layer_activations=False,
    show_trainable=True
)

In [ ]:
epochs=12
batch_size=32
steps=len(train)//batch_size

for i in range(epochs):
    generator=data_generator(train,mapping,features,tokenizer,max_length,vocab_size,batch_size)
    model.fit(generator, epochs=1,steps_per_epoch=steps,verbose=1)

In [ ]:
def convert_to_word(number, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == number:
            return word
    return None


def predict_caption(model, image, tokenizer, max_length):
    in_text = 'startseq'
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], max_length)
        y_pred = model.predict([image, sequence], verbose=0)
        y_pred = np.argmax(y_pred)
        word = convert_to_word(y_pred, tokenizer)
        if word is None:
            break
        in_text += " " + word
        if word == 'endseq':
            break

    return in_text

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
def generate_caption(image_name):
    image_id = image_name.split('.')[0]
    img_path = os.path.join(Base_dir, "Images", image_name)
    image = Image.open(img_path)
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)
    print(y_pred)
    plt.imshow(image)

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
actual, predicted = [] , []

for key in tqdm(test):

    captions = mapping[key]
    y_pred = predict_caption(model, features[key], tokenizer, max_length)
    actual_captions = [caption.split() for caption in captions]
    y_pred = y_pred.split()
    actual.append(actual_captions)
    predicted.append(y_pred)
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

In [ ]:
model.save('model.h5')
pickle.dump(tokenizer , open('tokenizer.pkl','wb'))

In [ ]:
!pip install -q gradio
import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

# Helper for prediction
def predict_caption_gradio(img):
    # Preprocess image for VGG16
    img = img.resize((224, 224))
    img_array = img_to_array(img)
    img_array = img_array.reshape((1, img_array.shape[0], img_array.shape[1], img_array.shape[2]))
    img_array = preprocess_input(img_array)

    # Extract features using the existing vgg_model in memory
    feature = vgg_model.predict(img_array, verbose=0)

    # Generate caption using the existing predict_caption function
    caption = predict_caption(model, feature, tokenizer, max_length)

    # Clean up the output tags
    return caption.replace('startseq ', '').replace(' endseq', '')

# Create Gradio Interface
interface = gr.Interface(
    fn=predict_caption_gradio,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=gr.Textbox(label="Generated Caption"),
    title="ဠး Image Caption Generator",
    description="Upload an image and the trained model will describe it!"
)

# Launch with share=True to get a public URL
interface.launch(share=True)